In [1]:
import os
from dotenv import load_dotenv

load_dotenv() 

API_KEY = os.getenv("FRED_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "FRED_API_KEY not found. Create a .env file or set an environment variable."
    )

In [2]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key=API_KEY)

START = '1986-01-01'
END   = '2026-02-01'

# --- Macro series (monthly) ---
series_monthly = {
    'CPI':            'CPIAUCSL',
    'TB3MS':          'TB3MS',
    'M1':             'M1SL', # changed definition in 2020
    'M2':             'M2SL',
}

# --- Exchange rates (daily → resample to monthly) ---
series_fx = {
    'AUD_USD': 'DEXUSAL',
    'CAD_USD': 'DEXCAUS',
    'NZD_USD': 'DEXUSNZ',
    'ZAR_USD': 'DEXSFUS', # only starts 1994
}

# Pull monthly macro
macro_df = pd.DataFrame({
    name: fred.get_series(sid, observation_start=START, observation_end=END)
    for name, sid in series_monthly.items()
})

# Pull FX and resample to monthly mean
fx_df = pd.DataFrame({
    name: fred.get_series(sid, observation_start=START, observation_end=END)
           .resample('MS').mean()          # Month-start, mean of daily values
    for name, sid in series_fx.items()
})

# Combine
df = macro_df.join(fx_df, how='outer')
df.index = pd.to_datetime(df.index)
df = df.resample('MS').last()             # Align all to month-start frequency

print(df.head())
print(df.tail())

              CPI  TB3MS     M1      M2   AUD_USD   CAD_USD   NZD_USD  \
1986-01-01  109.9   7.07  621.4  2502.1  0.700005  1.406981  0.516571   
1986-02-01  109.7   7.06  625.2  2512.9  0.699284  1.404284  0.531774   
1986-03-01  109.1   6.56  633.5  2533.1  0.707938  1.400910  0.528200   
1986-04-01  108.7   6.06  641.0  2557.8  0.722841  1.387868  0.561268   
1986-05-01  109.0   6.15  652.0  2584.8  0.727195  1.375719  0.566657   

             ZAR_USD  
1986-01-01  2.362762  
1986-02-01  2.089716  
1986-03-01  2.040943  
1986-04-01  2.051609  
1986-05-01  2.194029  
                CPI  TB3MS       M1       M2   AUD_USD   CAD_USD   NZD_USD  \
2025-10-01      NaN   3.82  18986.0  22250.4  0.654682  1.398768  0.576491   
2025-11-01  325.063   3.78  19022.7  22296.5  0.650172  1.405617  0.565206   
2025-12-01  326.031   3.59  19100.8  22386.9  0.664264  1.379491  0.578691   
2026-01-01  326.588   3.57  19201.1  22469.1  0.679175  1.377080  0.585090   
2026-02-01  327.460   3.60  19396

In [3]:
df.index.name = 'Date'
df.tail()

,CPI,TB3MS,M1,M2,AUD_USD,CAD_USD,NZD_USD,ZAR_USD
Date,,,,,,,,
2025-10-01,NaN,3.82,18986.0,22250.4,0.654682,1.398768,0.576491,17.272164
2025-11-01,325.063,3.78,19022.7,22296.5,0.650172,1.405617,0.565206,17.229661
2025-12-01,326.031,3.59,19100.8,22386.9,0.664264,1.379491,0.578691,16.832650
2026-01-01,326.588,3.57,19201.1,22469.1,0.679175,1.377080,0.585090,16.262495
2026-02-01,327.460,3.60,19396.9,22667.3,NaN,NaN,NaN,NaN


In [4]:
df.to_csv("predictor_data.csv")